In [ ]:
import numpy as np
from PIL import Image

# -------------------------------
# Parámetros
# -------------------------------

GRID_SIZE = 8
NUM_CANDIDATES = 5000
NUM_SELECTED = 5
CELL_SIZE = 40

# pesos del score
W_TRANSITIONS = 1.0
W_CORNERS = 3.0
W_SYMMETRY = -20.0


# -------------------------------
# Generación
# -------------------------------

def generate_candidate(size):
    """
    Genera un patrón binario aleatorio con aproximadamente
    50% de celdas negras.
    """
    return np.random.randint(0, 2, (size, size), dtype=np.uint8)


# -------------------------------
# Métricas
# -------------------------------

def count_transitions(pattern):
    """
    Cuenta cambios horizontal y vertical.
    """
    horizontal = np.sum(pattern[:, :-1] != pattern[:, 1:])
    vertical = np.sum(pattern[:-1, :] != pattern[1:, :])
    return horizontal + vertical


def count_corners(pattern):
    """
    Cuenta esquinas tipo checkerboard.
    """

    count = 0

    for y in range(pattern.shape[0]-1):
        for x in range(pattern.shape[1]-1):

            block = pattern[y:y+2, x:x+2]

            if np.array_equal(block, [[0,1],[1,0]]):
                count += 1

            elif np.array_equal(block, [[1,0],[0,1]]):
                count += 1

    return count


def symmetry_score(pattern):
    """
    Penaliza simetría.
    """

    score = 0

    if np.array_equal(pattern, np.flipud(pattern)):
        score += 1

    if np.array_equal(pattern, np.fliplr(pattern)):
        score += 1

    if np.array_equal(pattern, np.rot90(pattern)):
        score += 1

    if np.array_equal(pattern, np.rot90(pattern,2)):
        score += 1

    return score


def score(pattern):

    t = count_transitions(pattern)
    c = count_corners(pattern)
    s = symmetry_score(pattern)

    return (
        W_TRANSITIONS*t +
        W_CORNERS*c +
        W_SYMMETRY*s
    )


# -------------------------------
# Render
# -------------------------------

def save_pattern(pattern, filename):

    h, w = pattern.shape

    img = Image.new(
        "L",
        (w*CELL_SIZE, h*CELL_SIZE),
        color=255
    )

    pixels = img.load()

    for y in range(h):
        for x in range(w):

            value = 0 if pattern[y,x] else 255

            for yy in range(CELL_SIZE):
                for xx in range(CELL_SIZE):
                    pixels[
                        x*CELL_SIZE+xx,
                        y*CELL_SIZE+yy
                    ] = value

    img.save(filename)


# -------------------------------
# Selección
# -------------------------------

best = []

for _ in range(NUM_CANDIDATES):

    p = generate_candidate(GRID_SIZE)

    best.append((score(p), p))

best.sort(key=lambda x: x[0], reverse=True)

selected = []

for _, pattern in best:

    keep = True

    for old in selected:

        similarity = np.mean(pattern == old)

        if similarity > 0.80:
            keep = False
            break

    if keep:
        selected.append(pattern)

    if len(selected) == NUM_SELECTED:
        break


for i, pattern in enumerate(selected):

    save_pattern(pattern, f"marker_{i}.png")

print("Generados", len(selected), "marcadores.")